# ДЗ 31. Прогнозирование загрузки метро (Metro Interstate Traffic Volume)

Цель — построить модель, предсказывающую объём трафика (`traffic_volume`) на ближайшую неделю вперёд (24*7 точек), используя данные [Metro Interstate Traffic Volume Dataset](https://archive.ics.uci.edu/ml/datasets/Metro+Interstate+Traffic+Volume).

Несмотря на название датасета (это на самом деле данные о трафике на шоссе I-94 между Миннеаполисом и Сент-Полом), в задании он используется как прокси для загрузки метро — методология прогнозирования от этого не меняется.

План работы:
1. Загрузка данных и базовый EDA
2. Чистка ряда: дубликаты, выравнивание временной сетки, интерполяция пропусков
3. Отбор релевантной истории + генерация признаков (день недели, час)
4. Baseline-прогноз (среднее по `weekday`/`hour`)
5. Модели: SARIMA, SARIMAX с температурой, градиентный бустинг на признаках
6. Финальный прогноз на следующую неделю с доверительным интервалом


## 1. Загрузка данных и базовый EDA

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42


In [ ]:
raw = pd.read_csv("data/Metro_Interstate_Traffic_Volume.csv")
print(raw.shape)
raw.head()


In [ ]:
# По заданию нужны только 4 столбца
df = raw[["date_time", "holiday", "temp", "traffic_volume"]].copy()
df["date_time"] = pd.to_datetime(df["date_time"])
df.info()


In [ ]:
df.describe(include="all")


### Пропуски и базовые аномалии


In [ ]:
print("Пропуски по столбцам:")
print(df.isna().sum())
print()
print("Уникальные значения holiday:", df["holiday"].nunique())
print(df["holiday"].value_counts().head(15))


In [ ]:
# temp = 0 Kelvin выглядит как явный выброс/ошибка датчика (0K это abs. zero, нереально)
print("Число записей с temp == 0:", (df["temp"] == 0).sum())
df.loc[df["temp"] == 0, "temp"] = np.nan


### Дубликаты по времени

В датасете каждая строка соответствует часу, но часто на один и тот же `date_time` приходится
несколько записей (например, из-за нескольких значений `weather_description` для одного часа
в исходных данных NOAA). Так как нам нужны только 4 колонки, такие дубликаты по `date_time`
надо схлопнуть.


In [ ]:
dup_mask = df.duplicated(subset="date_time", keep=False)
print(f"Строк с повторяющимся date_time: {dup_mask.sum()} из {len(df)}")
df[dup_mask].sort_values("date_time").head(6)


In [ ]:
# Убираем полные дубликаты, а затем схлопываем оставшиеся дубликаты по date_time
# (берём первую запись - traffic_volume/temp в пределах часа отличаются незначительно)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="date_time", keep="first")
df = df.sort_values("date_time").reset_index(drop=True)
print(df.shape)


### Равномерность временной сетки

Проверим, действительно ли все точки идут с шагом в 1 час.


In [ ]:
diffs = df["date_time"].diff().dropna()
print(diffs.value_counts().head(10))
print()
print("Доля шагов != 1 час:", (diffs != pd.Timedelta(hours=1)).mean())


In [ ]:
gap_sizes = diffs[diffs > pd.Timedelta(hours=1)].sort_values(ascending=False)
print("Крупнейшие пропуски во времени:")
print(gap_sizes.head(10))
print()
print(f"Всего 'дыр' длиннее 1 часа: {len(gap_sizes)}")
print(f"Суммарно пропущено часов: {(gap_sizes / pd.Timedelta(hours=1) - 1).sum():.0f}")


Есть как дубликаты, так и заметные пропуски во времени (вплоть до многодневных). Выровняем сетку
до строго часовой при помощи `pd.date_range`, переиндексируем и заполним пропуски линейной
интерполяцией.


In [ ]:
df = df.set_index("date_time")

full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
print(f"Было точек: {len(df)}, должно быть при равномерной сетке: {len(full_index)}")

df = df.reindex(full_index)
df.index.name = "date_time"


In [ ]:
# holiday - категориальный признак, интерполировать нельзя - заполняем "None" (нет праздника)
df["holiday"] = df["holiday"].fillna("None")

# temp и traffic_volume - непрерывные ряды, линейная интерполяция
df["temp"] = df["temp"].interpolate(method="linear")
df["traffic_volume"] = df["traffic_volume"].interpolate(method="linear")

print(df.isna().sum())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df["traffic_volume"], lw=0.5)
axes[0].set_title("traffic_volume, весь период")
axes[1].plot(df.index, df["temp"], lw=0.5, color="tab:orange")
axes[1].set_title("temp, весь период")
plt.tight_layout()
plt.show()


### Недельная и суточная сезонность

Посмотрим на характерный профиль загрузки по дням недели и часам на небольшом фрагменте ряда.


In [ ]:
sample = df.loc["2018-08-01":"2018-08-21"]
plt.figure(figsize=(14, 4))
plt.plot(sample.index, sample["traffic_volume"])
plt.title("traffic_volume, 3 недели в августе 2018")
plt.show()


In [ ]:
tmp = df.copy()
tmp["weekday"] = tmp.index.weekday
tmp["hour"] = tmp.index.hour

pivot = tmp.groupby(["weekday", "hour"])["traffic_volume"].mean().unstack()
plt.figure(figsize=(14, 5))
plt.imshow(pivot, aspect="auto", cmap="viridis")
plt.colorbar(label="mean traffic_volume")
plt.yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.xlabel("hour of day")
plt.title("Средняя загрузка по дню недели и часу")
plt.show()


Видно чёткий будний профиль с двумя пиками (утро/вечер) и заметно более низкую, растянутую по
времени загрузку по выходным. Это подтверждает, что стоит использовать недельную сезонность
(24*7 часов) при моделировании.


## 2. Train/test split и отбор релевантной истории

Отложим последние 2 недели ряда для теста, а из оставшейся истории решим, сколько данных
реально нужно для обучения.


In [ ]:
TEST_HOURS = 24 * 7 * 2  # 2 недели

train_full = df.iloc[:-TEST_HOURS].copy()
test = df.iloc[-TEST_HOURS:].copy()

print("train_full:", train_full.index.min(), "-", train_full.index.max(), f"({len(train_full)} точек)")
print("test:      ", test.index.min(), "-", test.index.max(), f"({len(test)} точек)")


In [ ]:
# Посмотрим на весь train, разбитый по годам - есть ли структурные сдвиги уровня трафика
yearly = train_full.copy()
yearly["year"] = yearly.index.year
fig, ax = plt.subplots(figsize=(14, 4))
for year, grp in yearly.groupby("year"):
    ax.plot(grp.index.dayofyear, grp["traffic_volume"], alpha=0.5, lw=0.6, label=str(year))
ax.legend(title="year", ncol=6, fontsize=8)
ax.set_xlabel("day of year")
ax.set_title("traffic_volume по дням года, наложение по годам")
plt.show()


In [ ]:
train_full.groupby(train_full.index.year)["traffic_volume"].agg(["mean", "std", "count"])


В данных есть многомесячный разрыв в 2014-2015 (устройство сбора данных не работало), а также
заметно неполный 2012 и 2013 год. Общий уровень и недельно-суточный профиль трафика в
последние ~2 года выглядит стабильным (нет выраженного тренда), поэтому для обучения моделей
возьмём только последние 2 полных года истории перед тестовым отрезком - этого достаточно,
чтобы модель увидела все сезонные эффекты (включая праздники/зиму/лето), и не тратим время на
обучение по устаревшим и разреженным данным 2012-2015 годов.


In [ ]:
TRAIN_YEARS = 2
train_start = test.index.min() - pd.DateOffset(years=TRAIN_YEARS)
train = train_full.loc[train_full.index >= train_start].copy()

print("train:", train.index.min(), "-", train.index.max(), f"({len(train)} точек)")


## 3. Генерация признаков

In [ ]:
def add_time_features(frame):
    out = frame.copy()
    out["hour"] = out.index.hour
    out["weekday"] = out.index.weekday  # 0=Mon ... 6=Sun
    out["is_weekend"] = out["weekday"].isin([5, 6]).astype(int)
    out["month"] = out.index.month
    out["is_holiday"] = (out["holiday"] != "None").astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["weekday_sin"] = np.sin(2 * np.pi * out["weekday"] / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * out["weekday"] / 7)
    return out

train = add_time_features(train)
test = add_time_features(test)
train.head()


## 4. Baseline: среднее по (weekday, hour)

In [ ]:
baseline_table = train.groupby(["weekday", "hour"])["traffic_volume"].mean()
baseline_table.head(10)


In [ ]:
def predict_baseline(frame, table):
    return frame.set_index(["weekday", "hour"]).index.map(table)

test["baseline_pred"] = predict_baseline(test.reset_index(), baseline_table).values
test[["traffic_volume", "baseline_pred"]].head()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

def report_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"{name:30s}  MAE={mae:8.1f}  RMSE={rmse:8.1f}  MAPE={mape:6.2f}%")
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape}

results = []
results.append(report_metrics(test["traffic_volume"], test["baseline_pred"], "baseline (weekday/hour mean)"))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test.index, test["traffic_volume"], label="actual")
ax.plot(test.index, test["baseline_pred"], label="baseline pred", alpha=0.8)
ax.legend()
ax.set_title("Baseline прогноз vs факт (2 недели теста)")
plt.show()


## 5. Модели

### 5.1 SARIMA с недельной сезонностью (24*7)

SARIMA на почасовых данных с сезонным периодом 24*7 = 168 достаточно тяжёлая по памяти/времени,
поэтому подберём компактный порядок и обучим на last-N наблюдениях трейна (последние несколько
недель) - этого достаточно, чтобы модель захватила суточно-недельный профиль, не взрываясь по
ресурсам.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

SEASONAL_PERIOD = 24 * 7
# SARIMA с сезонным периодом 168 на почасовых данных - дорогая по времени модель:
# state-space представление растёт с числом сезонных AR/MA лагов, поэтому сезонную
# MA-часть отключаем (seasonal_order=(1,0,0,168) - только сезонный AR), ограничиваем
# обучающую выборку 6 неделями. Этого достаточно, чтобы захватить недельный профиль,
# при этом обучение занимает ~1-2 минуты вместо часов.
SARIMA_TRAIN_WEEKS = 4
sarima_train = train["traffic_volume"].iloc[-SEASONAL_PERIOD * SARIMA_TRAIN_WEEKS:]

print(f"SARIMA train size: {len(sarima_train)} точек (~{len(sarima_train)/24:.0f} дней)")


In [ ]:
sarima_model = SARIMAX(
    sarima_train,
    order=(1, 0, 0),
    seasonal_order=(1, 0, 0, SEASONAL_PERIOD),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False, maxiter=50)
print(sarima_fit.summary())


In [ ]:
sarima_forecast = sarima_fit.get_forecast(steps=TEST_HOURS)
sarima_pred = sarima_forecast.predicted_mean
sarima_ci = sarima_forecast.conf_int(alpha=0.05)

test["sarima_pred"] = sarima_pred.values
results.append(report_metrics(test["traffic_volume"], test["sarima_pred"], "SARIMA(1,0,1)(1,1,1,168)"))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test.index, test["traffic_volume"], label="actual")
ax.plot(test.index, test["sarima_pred"], label="SARIMA pred", alpha=0.8)
ax.legend()
ax.set_title("SARIMA прогноз vs факт")
plt.show()


### 5.2 SARIMAX с температурой как экзогенным признаком

Отдельно прогнозируем температуру на тестовый период (тоже через SARIMA с суточной
сезонностью), а затем используем эти предсказания температуры как экзогенную переменную
для прогноза трафика.


In [ ]:
temp_train = train["temp"].iloc[-24 * 30:]  # последний месяц для сезонности температуры

temp_model = SARIMAX(
    temp_train,
    order=(2, 0, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
temp_fit = temp_model.fit(disp=False)
temp_forecast = temp_fit.get_forecast(steps=TEST_HOURS).predicted_mean
temp_forecast.index = test.index

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(test.index, test["temp"], label="actual temp")
ax.plot(test.index, temp_forecast, label="forecast temp", alpha=0.8)
ax.legend()
ax.set_title("Прогноз температуры на тестовый период")
plt.show()


In [ ]:
sarimax_train_exog = train[["traffic_volume", "temp"]].iloc[-SEASONAL_PERIOD * SARIMA_TRAIN_WEEKS:]

sarimax_model = SARIMAX(
    sarimax_train_exog["traffic_volume"],
    exog=sarimax_train_exog[["temp"]],
    order=(1, 0, 0),
    seasonal_order=(1, 0, 0, SEASONAL_PERIOD),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_fit = sarimax_model.fit(disp=False, maxiter=50)

sarimax_pred = sarimax_fit.get_forecast(steps=TEST_HOURS, exog=temp_forecast.values.reshape(-1, 1)).predicted_mean
test["sarimax_pred"] = sarimax_pred.values
results.append(report_metrics(test["traffic_volume"], test["sarimax_pred"], "SARIMAX + forecast(temp)"))


### 5.3 Градиентный бустинг на календарных признаках + температуре

Табличная модель: используем только признаки, известные заранее (час, день недели, месяц,
праздник, температура), без лагов таргета - это честная постановка для прогноза на 7 дней
вперёд одним шагом (multi-step direct forecasting).


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

feature_cols = ["hour", "weekday", "is_weekend", "month", "is_holiday",
                 "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "temp"]

gbr = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=RANDOM_STATE,
)
gbr.fit(train[feature_cols], train["traffic_volume"])


In [ ]:
test_features_for_gbr = test[feature_cols].copy()
test_features_for_gbr["temp"] = temp_forecast.values  # честно используем прогноз температуры, не факт

test["gbr_pred"] = gbr.predict(test_features_for_gbr)
results.append(report_metrics(test["traffic_volume"], test["gbr_pred"], "GradientBoosting (calendar + temp forecast)"))


In [ ]:
importances = pd.Series(gbr.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind="bar", figsize=(10, 3), title="Feature importances (GBR)")
plt.tight_layout()
plt.show()
